# 03 — Repetition penalty = 1.1 — Qwen3-1.7B fine-tuned

Neste teste, o modelo ajustado Qwen3-1.7B é avaliado novamente sobre os mesmos **215 autores de teste**, alterando apenas o parâmetro de geração:

- `repetition_penalty = 1.1`

Os demais parâmetros de inferência são mantidos conforme o experimento original:

- `do_sample = True`
- `temperature = 0.7`
- `top_p = 0.8`
- `top_k = 20`
- `min_p = 0.0`
- `enable_thinking = False`
- `max_input_tokens = 8192`
- `max_new_tokens = 1024`
- `batch_size = 4`
- `seed = 42`

Como `repetition_penalty` é utilizado somente na geração, este notebook **reutiliza o adaptador QLoRA já treinado** no experimento principal e não executa novamente o treinamento.

A avaliação mantém o mesmo protocolo: SBERT, matching global greedy 1-para-1, threshold 0,75 e as mesmas métricas.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np

current = Path.cwd().resolve()
candidates = [current, *current.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src").exists()), current)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Raiz do projeto: {PROJECT_ROOT}")

## 1. Caminhos

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"
GROUND_TRUTH_DIR = DATA_DIR / "ground_truth"
SPLITS_DIR = DATA_DIR / "splits"

SHARED_DATA_DIR = PROJECT_ROOT / "results" / "finetuning" / "shared_data"
BASE_EXPERIMENT_DIR = PROJECT_ROOT / "results" / "finetuning" / "qwen3_1_7b"
ADAPTER_DIR = BASE_EXPERIMENT_DIR / "adapter_qlora_final"

OUTPUT_DIR = PROJECT_ROOT / "results" / "additional_analysis" / "repetition_penalty_1_1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_FILE = SPLITS_DIR / "split_autores_seed42.json"
QRELS_FILE = GROUND_TRUTH_DIR / "LExR-prof-qrels_filtrado"
DOCUMENT_PROFILES_FILE = DATA_DIR / "perfis_documento_qwen.json"

TAGS_FILE = OUTPUT_DIR / "tags_brutas_qwen3_1_7b_rep_penalty_1_1.json"
RANKING_FILE = OUTPUT_DIR / "ranking_qwen3_1_7b_rep_penalty_1_1.json"
CHECKPOINT_FILE = OUTPUT_DIR / "checkpoint_qwen3_1_7b_rep_penalty_1_1.json"

METRICS_FILE = OUTPUT_DIR / "metricas_por_autor_qwen3_1_7b_rep_penalty_1_1.csv"
MATCHING_FILE = OUTPUT_DIR / "avaliacoes_gerais_qwen3_1_7b_rep_penalty_1_1.csv"
SIM_DIR = OUTPUT_DIR / "sim_matrices"

required = {
    "dataset de teste": SHARED_DATA_DIR / "dataset_teste_215_autores.jsonl",
    "split": SPLIT_FILE,
    "adaptador QLoRA": ADAPTER_DIR / "adapter_config.json",
    "qrels": QRELS_FILE,
    "perfis documentais": DOCUMENT_PROFILES_FILE,
}

for name, path in required.items():
    print(f"{name:24s} -> {'OK' if path.exists() else 'não encontrado'}")

## 2. Inferência com repetition penalty = 1.1

A inferência reutiliza o adaptador produzido no experimento principal do Qwen3-1.7B.

O único parâmetro experimental alterado nesta análise é `repetition_penalty`, definido como `1.1`.

In [ ]:
cmd = [
    sys.executable, "-m", "src.finetuning.finetuned_inference",
    "--data-dir", str(SHARED_DATA_DIR),
    "--split-path", str(SPLIT_FILE),
    "--model", "Qwen/Qwen3-1.7B",
    "--adapter", str(ADAPTER_DIR),
    "--output", str(TAGS_FILE),
    "--ranking-output", str(RANKING_FILE),
    "--checkpoint", str(CHECKPOINT_FILE),
    "--expected-test", "215",
    "--seed", "42",
    "--batch-size", "4",
    "--checkpoint-every-authors", "12",
    "--n-tags", "30",
    "--max-input-tokens", "8192",
    "--max-new-tokens", "1024",
    "--do-sample",
    "--temperature", "0.7",
    "--top-p", "0.8",
    "--top-k", "20",
    "--min-p", "0.0",
    "--repetition-penalty", "1.1",
]

subprocess.run(cmd, check=True)

## 3. Avaliação semântica

In [ ]:
from sentence_transformers import SentenceTransformer

from src.evaluation.semantic_matching import (
    carregar_qrels,
    construir_vocabulario,
    encodar_com_sbert,
    construir_matriz_similaridade,
    matching_greedy_1_to_1,
    salvar_npz,
    salvar_csv_avaliacoes_gerais,
)
from src.evaluation.metrics import (
    carregar_perfis_por_documento,
    avaliar_autor,
    salvar_csv_metricas,
)

SBERT_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
THRESHOLD = 0.75
THRESHOLD_COVERAGE = 0.75
TOP_K = 20

with RANKING_FILE.open("r", encoding="utf-8") as f:
    rankings_json = json.load(f)

rankings = {
    str(author): [(str(tag), float(score)) for tag, score in ranking]
    for author, ranking in rankings_json.items()
}

with SPLIT_FILE.open("r", encoding="utf-8") as f:
    split = json.load(f)

autores_teste = [str(a) for a in split["test"]]

if len(autores_teste) != 215:
    raise ValueError(f"O split de teste deveria conter 215 autores, mas contém {len(autores_teste)}.")

if set(rankings) != set(autores_teste):
    raise ValueError("O ranking produzido não corresponde exatamente aos 215 autores do teste.")

gt_norm_global, gt_original_global = carregar_qrels(QRELS_FILE)
perfis_doc_global = carregar_perfis_por_documento(DOCUMENT_PROFILES_FILE)

gt_norm = {a: gt_norm_global[a] for a in autores_teste}
gt_original = {a: gt_original_global[a] for a in autores_teste}
perfis_doc = {a: perfis_doc_global.get(a, {}) for a in autores_teste}

modelo_sbert = SentenceTransformer(SBERT_MODEL)

vocabulario = construir_vocabulario(
    rankings,
    gt_norm,
    perfis_doc,
    top_k_pred=TOP_K,
)

cache_emb = encodar_com_sbert(
    modelo_sbert,
    vocabulario,
    batch_size=256,
)

## 4. Matching e métricas

In [ ]:
dados_por_autor = {}
matching_por_autor = {}
metricas_por_autor = {}

SIM_DIR.mkdir(parents=True, exist_ok=True)

for autor in autores_teste:
    dados = construir_matriz_similaridade(
        autor,
        rankings[autor],
        gt_norm[autor],
        cache_emb,
        top_k=TOP_K,
    )

    if dados is None:
        continue

    matched_weights, matched_idx, matched_sims = matching_greedy_1_to_1(
        dados["sim"],
        dados["gold_weights"],
        theta=THRESHOLD,
    )

    salvar_npz(
        dados,
        matched_idx,
        matched_weights,
        matched_sims,
        SIM_DIR,
    )

    dados_por_autor[autor] = dados
    matching_por_autor[autor] = {
        "matched_weights": matched_weights,
        "matched_idx": matched_idx,
        "matched_sims": matched_sims,
    }

    docs_autor = perfis_doc.get(autor, {})
    metricas_por_autor[autor] = avaliar_autor(
        dados,
        matched_weights,
        docs_autor,
        docs_autor,
        cache_emb,
        theta_cov=THRESHOLD_COVERAGE,
    )

print(f"Autores avaliados: {len(metricas_por_autor)}")

## 5. Salvamento dos resultados

In [ ]:
MODELO_LABEL = "Qwen3-1.7B Fine-tuning — repetition_penalty=1.1"

salvar_csv_metricas(
    metricas_por_autor,
    METRICS_FILE,
    modelo=MODELO_LABEL,
)

salvar_csv_avaliacoes_gerais(
    dados_por_autor,
    matching_por_autor,
    gt_original,
    MATCHING_FILE,
    modelo=MODELO_LABEL,
    rank_max=20,
)

print(f"Métricas: {METRICS_FILE}")
print(f"Avaliações detalhadas: {MATCHING_FILE}")
print(f"Matrizes: {SIM_DIR}")

## 6. Médias das métricas

In [ ]:
if metricas_por_autor:
    nomes = list(next(iter(metricas_por_autor.values())).keys())
    medias = {
        nome: float(np.mean([m[nome] for m in metricas_por_autor.values()]))
        for nome in nomes
    }

    for nome, valor in medias.items():
        print(f"{nome:18s}: {valor:.4f}")